In [1]:
# kaggle_dict = {
#     "username": "danahatem202301816",
#     "key": "KGAT_afe007a68262f1fb99cd0f1e31ec0035"
# }

In [2]:
!pip install opendatasets --upgrade --quiet

import opendatasets as od
import os
import pandas as pd
import shutil

# --- DOWNLOAD PHASE ---
# Paste the URLs directly
ham_url = "https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000"
isic_url = "https://www.kaggle.com/datasets/nodoubttome/skin-cancer9-classesisic"

print("Downloading HAM10000...")
od.download(ham_url)
print("Downloading ISIC 9 Classes...")
od.download(isic_url)

# --- MERGE PHASE ---
master_dir = 'unified_dataset'
os.makedirs(master_dir, exist_ok=True)

# Mappings to standardize names
ham_mapping = {'nv': 'melanocytic_nevi', 'mel': 'melanoma', 'bcc': 'basal_cell_carcinoma',
               'akiec': 'actinic_keratosis', 'bkl': 'benign_keratosis', 'df': 'dermatofibroma',
               'vasc': 'vascular_lesion'}

isic_mapping = {'actinic keratosis': 'actinic_keratosis', 'basal cell carcinoma': 'basal_cell_carcinoma',
                'dermatofibroma': 'dermatofibroma', 'melanoma': 'melanoma', 'nevus': 'melanocytic_nevi',
                'pigmented benign keratosis': 'benign_keratosis', 'seborrheic keratosis': 'benign_keratosis',
                'squamous cell carcinoma': 'squamous_cell_carcinoma', 'vascular lesion': 'vascular_lesion'}

# Process HAM10000
ham_base = 'skin-cancer-mnist-ham10000'
metadata = pd.read_csv(f'{ham_base}/HAM10000_metadata.csv')
ham_folders = [f'{ham_base}/ham10000_images_part_1', f'{ham_base}/ham10000_images_part_2']

for _, row in metadata.iterrows():
    label = ham_mapping[row['dx']]
    dest = os.path.join(master_dir, label)
    os.makedirs(dest, exist_ok=True)
    img_name = row['image_id'] + '.jpg'
    for folder in ham_folders:
        src = os.path.join(folder, img_name)
        if os.path.exists(src):
            new_name = f"ham_{img_name}"
            shutil.copy(src, os.path.join(dest, new_name))
            break

# Process ISIC 9
isic_root = 'skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration'
for split in ['Train', 'Test']:
    split_path = os.path.join(isic_root, split)
    if not os.path.exists(split_path): continue
    for folder_name in os.listdir(split_path):
        if folder_name in isic_mapping:
            target_label = isic_mapping[folder_name]
            dest_folder = os.path.join(master_dir, target_label)
            os.makedirs(dest_folder, exist_ok=True)
            src_folder = os.path.join(split_path, folder_name)
            for img in os.listdir(src_folder):
                new_name = f"isic_{img}"
                shutil.copy(os.path.join(src_folder, img), os.path.join(dest_folder, new_name))

print(f"Done! Your combined dataset is ready in the '{master_dir}' folder.")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: danahatem202301816
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000


100%|██████████| 5.20G/5.20G [00:59<00:00, 93.7MB/s]



Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: danahatem202301816
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/nodoubttome/skin-cancer9-classesisic


100%|██████████| 786M/786M [00:08<00:00, 91.9MB/s]



Done! Your combined dataset is ready in the 'unified_dataset' folder.


In [3]:
import os
import pandas as pd

data_dir = "unified_dataset"

data = []

for label in os.listdir(data_dir):
    label_path = os.path.join(data_dir, label)

    if os.path.isdir(label_path):
        for img_name in os.listdir(label_path):
            img_path = os.path.join(label_path, img_name)
            data.append([img_path, label])

df = pd.DataFrame(data, columns=["path", "label"])

print("Dataset shape:", df.shape)
df.head()


Dataset shape: (12372, 2)


,path,label
0,unified_dataset/benign_keratosis/ham_ISIC_0030...,benign_keratosis
1,unified_dataset/benign_keratosis/ham_ISIC_0024...,benign_keratosis
2,unified_dataset/benign_keratosis/isic_ISIC_002...,benign_keratosis
3,unified_dataset/benign_keratosis/isic_ISIC_002...,benign_keratosis
4,unified_dataset/benign_keratosis/isic_ISIC_002...,benign_keratosis


In [4]:
import os
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["label_encoded"] = le.fit_transform(df["label"])

df.head()

,path,label,label_encoded
0,unified_dataset/benign_keratosis/ham_ISIC_0030...,benign_keratosis,2
1,unified_dataset/benign_keratosis/ham_ISIC_0024...,benign_keratosis,2
2,unified_dataset/benign_keratosis/isic_ISIC_002...,benign_keratosis,2
3,unified_dataset/benign_keratosis/isic_ISIC_002...,benign_keratosis,2
4,unified_dataset/benign_keratosis/isic_ISIC_002...,benign_keratosis,2


In [5]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df["label_encoded"], random_state=42
)


train_paths = set(train_df["path"])
test_paths = set(test_df["path"])

intersection = train_paths.intersection(test_paths)

print("Leakage samples:", len(intersection))

Leakage samples: 0


In [6]:
from PIL import Image
import numpy as np

def load_image(path, size=(224, 224)):
    img = Image.open(path).convert("RGB")  # IMPORTANT FIX
    img = img.resize(size)
    img = np.array(img) / 255.0
    return img


# Class distribution
print(df["label"].value_counts())

label
melanocytic_nevi           7078
benign_keratosis           1657
melanoma                   1567
basal_cell_carcinoma        906
actinic_keratosis           457
vascular_lesion             284
dermatofibroma              226
squamous_cell_carcinoma     197
Name: count, dtype: int64


In [7]:
df.to_csv("unified_dataset.csv", index=False)
train_df.to_csv("train.csv", index=False)
test_df.to_csv("test.csv", index=False)

In [8]:
train_df.to_csv("train.csv", index=False)
test_df.to_csv("test.csv", index=False)

In [9]:
from google.colab import drive
drive.mount('/content/drive')

train_df.to_csv("/content/drive/MyDrive/train.csv", index=False)
test_df.to_csv("/content/drive/MyDrive/test.csv", index=False)

print("Saved train and test CSVs to Drive")

Mounted at /content/drive
Saved train and test CSVs to Drive


In [13]:
# !cp -r /content/unified_dataset /content/drive/MyDrive/

In [11]:
import os

print(os.path.exists("/content/drive/MyDrive/train.csv"))
print(os.path.exists("/content/drive/MyDrive/test.csv"))

True
True


In [12]:
print("Total dataset size:", len(df))
print("Number of classes:", df["label"].nunique())

print("\nClass distribution:")
print(df["label"].value_counts())

Total dataset size: 12372
Number of classes: 8

Class distribution:
label
melanocytic_nevi           7078
benign_keratosis           1657
melanoma                   1567
basal_cell_carcinoma        906
actinic_keratosis           457
vascular_lesion             284
dermatofibroma              226
squamous_cell_carcinoma     197
Name: count, dtype: int64
